# Baseline evaluation

**Notebook 02 · Measure the original student before training**

Notebook 01 checked how a student response can receive teacher feedback and one
weight update. Now we need a starting score: how well does the **original,
unchanged student** answer several questions?

We load a fresh `Qwen/Qwen3-0.6B-Base`, ask a fixed set of arithmetic questions,
check its final answers, and save the complete experiment. We do not reuse the
student updated in notebook 01. No teacher, KL calculation, optimizer, or training
is needed here.

| Experiment | Protocol |
| :--- | :--- |
| Model | Fresh `Qwen/Qwen3-0.6B-Base` checkpoint |
| Questions | 32 reproducibly selected GSM8K test examples |
| Prompt style | Plain text, zero-shot: no worked examples or chat template |
| Generation | Greedy decoding; at most 512 new tokens per question |
| Main metric | Correct final numeric answers divided by **all** questions |
| Additional checks | Answer-format success, token-limit stops, response lengths |
| Output | Downloadable ZIP with the protocol, raw responses, scores, and environment |

> **Run status:** This notebook is ready for its first Colab GPU run. There are no
> saved model results yet. Small local checks can verify the scoring rules but
> cannot tell us the student's accuracy.

### Notebook guide

1. [Prepare the runtime](#baseline-setup)
2. [Define the questions and scoring rules](#baseline-protocol)
3. [Load a fresh student](#baseline-model)
4. [Generate and score responses](#baseline-generation)
5. [Read the results](#baseline-metrics)
6. [Save and download the experiment](#baseline-artifacts)
7. [What we achieved](#baseline-recap)

### Before you run

Open this notebook in a **fresh Colab GPU runtime** and run from top to bottom.
An L4 or A100 can be used for this student-only experiment. Do not keep notebook
01's large teacher in the same runtime. Restarting clears its GPU memory.

Because you are not using Google Drive, **download the results ZIP and save the
executed notebook before leaving Colab**. Files in the temporary runtime can disappear.

This is a small, custom evaluation protocol, not an official GSM8K leaderboard
score or a reproduction of the paper's evaluation settings. Keep this question
set out of later training and hyperparameter tuning.


---

<a id="baseline-setup"></a>

## 1. Prepare the runtime

### 1.1 Install the dependencies

We use `uv` to install into the Python environment running this notebook.
`datasets` downloads the question set, and `pandas` displays and exports tables.
If Colab asks for a runtime restart after installation, restart and begin again.


In [ ]:
import importlib.util
import subprocess
import sys

# Bootstrap uv only when it is missing. pip is used just for that bootstrap.
if importlib.util.find_spec("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

# --python directs installation into this notebook's actual kernel environment.
# This experiment loads only the student; no quantization or training is needed.
subprocess.run(
    [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable,
     "torch", "transformers>=4.51.0,<5", "accelerate", "datasets>=3,<5",
     "pandas", "huggingface_hub"],
    check=True,
)


### 1.2 Check the GPU and set the experiment size

BF16 and FP16 are compact formats for model calculations. Token IDs remain
integers. We use BF16 when supported and FP16 otherwise.

The fixed seed selects the same questions when the dataset revision is the same.
Greedy decoding chooses the highest-probability next token, rather than sampling.
The seed is recorded, but it does not make different hardware or package versions
numerically identical. We save that environment too.


In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import random
import re
import shutil
import time
import zipfile
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

# Start a fresh runtime if notebook 01's student or teacher is still in memory.
assert "teacher" not in globals() and "student" not in globals(), (
    "Restart the runtime, then run notebook 02 from the top. Do not reuse notebook 01's models."
)
assert torch.cuda.is_available(), "Choose a GPU runtime in Colab before continuing."
baseline_device = torch.device("cuda:0")
gpu_properties = torch.cuda.get_device_properties(baseline_device)
bf16_supported = torch.cuda.is_bf16_supported()
baseline_dtype = torch.bfloat16 if bf16_supported else torch.float16

# These are the settings of this evaluation, not a training configuration.
student_model_id = "Qwen/Qwen3-0.6B-Base"
student_revision = "main"  # Resolved to an exact checkpoint commit below.
dataset_id = "openai/gsm8k"
dataset_config = "main"
dataset_split = "test"  # Held out from later training and tuning.
dataset_revision = "main"  # Also resolved and recorded as an exact commit.
evaluation_seed = 42
number_of_examples = 32
max_new_tokens = 512
download_results = True  # Colab will offer the completed ZIP for download.
random.seed(evaluation_seed)
torch.manual_seed(evaluation_seed)

print("GPU:", gpu_properties.name)
print("GPU memory (GiB):", round(gpu_properties.total_memory / 1024**3, 1))
print("BF16 supported:", bf16_supported)
print("Model precision:", baseline_dtype)
print("Evaluation questions:", number_of_examples)


---

<a id="baseline-protocol"></a>

## 2. Define the questions and scoring rules

### 2.1 Decide what counts as an answer

[GSM8K](https://huggingface.co/datasets/openai/gsm8k) contains arithmetic word
problems with reference solutions. Its reference answers end with `####` followed
by the final number, as described in the [original dataset repository](https://github.com/openai/grade-school-math).

We ask the student to use that same final-line format. For example:

```text
There are 10 pencils and 3 are sold, leaving 7.
#### 7
```

The checker accepts a single marked final answer. It handles signs, decimals,
valid thousands separators, and an optional dollar sign. It rejects missing,
ambiguous, or malformed answers. It does **not** guess that the last number in
unfinished reasoning is the answer. A formatting failure counts as incorrect in
the main accuracy, and we also report format success separately.

This strict rule measures both answering correctly and following the output
format. A base model may struggle with either. We will inspect those failures
instead of silently dropping them from the denominator.


In [ ]:
# Only the question is inserted here. Reference solutions and answers are never
# included in the input sent to the student. No chat template is used for this base model.
evaluation_prompt_template = (
    "Question: {question}\n\n"
    "Solve the problem step by step. End your response with a single final line "
    "in the format '#### <number>'.\n\nAnswer:\n"
)


def build_evaluation_prompt(question):
    """Create the exact plain-text input that later evaluations must reuse."""
    assert isinstance(question, str) and question.strip()
    return evaluation_prompt_template.format(question=question)


def select_evaluation_indices(dataset_size, sample_size, seed):
    """Choose unique held-out examples without depending on global random state."""
    assert 0 < sample_size <= dataset_size
    return sorted(random.Random(seed).sample(range(dataset_size), sample_size))


def extract_final_answer(response_text):
    """Return a Decimal for one marked final number, or None for a format failure."""
    nonempty_lines = [line.strip() for line in response_text.splitlines() if line.strip()]
    # Multiple marked answers can mean the model continued into another problem.
    # Reject ambiguity, and require the answer to be the last nonempty line.
    marked_lines = [line for line in nonempty_lines if line.startswith("####")]
    if not nonempty_lines or len(marked_lines) != 1:
        return None
    number_pattern = r"[+-]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?"
    answer_match = re.fullmatch(r"####\s*\$?\s*(" + number_pattern + r")", nonempty_lines[-1])
    if answer_match is None:
        return None
    # Decimal compares numbers exactly, without floating-point rounding artifacts.
    return Decimal(answer_match.group(1).replace(",", ""))


def summarize_baseline_results(result_rows):
    """Use every question in the denominator, including format failures."""
    assert result_rows
    assert len({row["example_id"] for row in result_rows}) == len(result_rows)
    total = len(result_rows)
    return {
        "number_of_examples": total,
        "number_correct": sum(row["is_correct"] for row in result_rows),
        "accuracy": sum(row["is_correct"] for row in result_rows) / total,
        "number_parsed": sum(row["answer_parsed"] for row in result_rows),
        "answer_parse_rate": sum(row["answer_parsed"] for row in result_rows) / total,
        "number_at_token_limit": sum(row["hit_token_limit"] for row in result_rows),
        "mean_response_tokens": sum(row["response_length"] for row in result_rows) / total,
    }


### 2.2 Test the checker before evaluating the model

These small tests use made-up text, not model predictions. They verify that the
checker can read valid final answers, reject unclear ones, and count formatting
failures as incorrect. A passing checker test is not a student accuracy result.


In [ ]:
# Equivalent numeric spellings should compare equally.
assert extract_final_answer("Reasoning first.\n#### 42") == Decimal("42")
assert extract_final_answer("#### $1,234.00") == Decimal("1234")
assert extract_final_answer("#### -2.5") == Decimal("-2.50")
assert extract_final_answer("#### +7\n\n") == Decimal("7")

# Do not award credit for a number merely mentioned in unfinished reasoning,
# invalid separators, multiple final answers, units, or text after the final line.
for invalid_response in [
    "", "The answer might be 42", "#### 1,23", "#### 4 pencils",
    "#### 7\n#### 8", "#### 7\nHere is another question.", "#### NaN",
]:
    assert extract_final_answer(invalid_response) is None

chosen_indices = select_evaluation_indices(100, 32, evaluation_seed)
assert chosen_indices == select_evaluation_indices(100, 32, evaluation_seed)
assert len(chosen_indices) == len(set(chosen_indices)) == 32

# One correct, one wrong, and one unparsed answer must give 1/3 accuracy.
test_rows = [
    {"example_id": "a", "is_correct": True, "answer_parsed": True,
     "hit_token_limit": False, "response_length": 10},
    {"example_id": "b", "is_correct": False, "answer_parsed": True,
     "hit_token_limit": False, "response_length": 20},
    {"example_id": "c", "is_correct": False, "answer_parsed": False,
     "hit_token_limit": True, "response_length": 30},
]
test_summary = summarize_baseline_results(test_rows)
assert test_summary["accuracy"] == 1 / 3
assert test_summary["answer_parse_rate"] == 2 / 3
assert test_summary["number_at_token_limit"] == 1
print("All baseline scoring and selection tests passed.")


### 2.3 Select and record the held-out questions

We resolve the dataset name to an exact Git commit, then select 32 unique test
examples with seed 42. Saving that revision and the original row indices makes
the selection repeatable. To reproduce this run later, replace `dataset_revision`
with the saved commit, rather than relying on a potentially newer `main`.

The reference answer is used only for checking the response afterward. It never
appears in the student's prompt. These test examples must remain outside our
later training data. Use a separate validation set if you need to tune settings.


In [ ]:
# Revision pinning prevents a later dataset update from silently changing this run.
hub_api = HfApi()
resolved_dataset_revision = hub_api.dataset_info(dataset_id, revision=dataset_revision).sha
assert resolved_dataset_revision
held_out_dataset = load_dataset(
    dataset_id, dataset_config, split=dataset_split,
    revision=resolved_dataset_revision,
)
evaluation_indices = select_evaluation_indices(
    len(held_out_dataset), number_of_examples, evaluation_seed,
)
evaluation_examples = []
for source_index in evaluation_indices:
    source_example = held_out_dataset[source_index]
    expected_answer = extract_final_answer(source_example["answer"])
    assert expected_answer is not None, "A reference answer could not be parsed."
    evaluation_examples.append({
        "example_id": f"gsm8k/{dataset_split}/{source_index}",
        "source_index": source_index,
        "question": source_example["question"],
        "prompt": build_evaluation_prompt(source_example["question"]),
        "expected_answer": str(expected_answer),
    })

assert len(evaluation_examples) == number_of_examples
assert len({example["example_id"] for example in evaluation_examples}) == number_of_examples
print("Dataset revision:", resolved_dataset_revision)
print("Split:", dataset_split)
print("Selected row indices:", evaluation_indices)
print("First prompt sent to the student:\n")
print(evaluation_examples[0]["prompt"])


---

<a id="baseline-model"></a>

## 3. Load a fresh student

We load the original checkpoint directly from Hugging Face, using its own
tokenizer and an exact resolved model revision. This gives us a starting point
before any of our project updates. It does not undo the model's original pretraining.

`eval()` turns off training-only behavior. Generation runs without gradients.
We load no teacher and create no optimizer.

For evaluation we choose **greedy decoding**: each step takes the token with the
highest probability. Notebook 01 sampled to demonstrate on-policy trajectories;
this notebook measures a fixed evaluation policy. Future before/after evaluations
must reuse this prompt, question set, token budget, and decoding policy.


In [ ]:
# Reloading this cell into the same kernel can hold a second model temporarily.
# Restart first if a previous baseline model is still loaded.
assert "baseline_student" not in globals(), "Restart the runtime before reloading the baseline model."
resolved_student_revision = hub_api.model_info(
    student_model_id, revision=student_revision,
).sha
assert resolved_student_revision
baseline_tokenizer = AutoTokenizer.from_pretrained(
    student_model_id, revision=resolved_student_revision,
)
if baseline_tokenizer.pad_token_id is None:
    baseline_tokenizer.pad_token = baseline_tokenizer.eos_token
assert baseline_tokenizer.pad_token_id is not None

baseline_student = AutoModelForCausalLM.from_pretrained(
    student_model_id, revision=resolved_student_revision,
    torch_dtype=baseline_dtype, device_map={"": 0},
)
baseline_student.eval()
assert len(baseline_tokenizer) <= baseline_student.config.vocab_size

# Build an explicit generation configuration instead of inheriting the model's
# default sampling settings. Temperature/top-k/top-p are unnecessary for greedy decoding.
baseline_generation_config = GenerationConfig(
    max_new_tokens=max_new_tokens,
    do_sample=False,
    num_beams=1,
    num_return_sequences=1,
    use_cache=True,
    pad_token_id=baseline_tokenizer.pad_token_id,
    eos_token_id=baseline_student.generation_config.eos_token_id,
)
if baseline_generation_config.eos_token_id is None:
    baseline_generation_config.eos_token_id = baseline_tokenizer.eos_token_id
eos_token_ids = baseline_generation_config.eos_token_id
if isinstance(eos_token_ids, int):
    eos_token_ids = [eos_token_ids]
eos_token_ids = set(eos_token_ids or [])
assert eos_token_ids, "An end-of-sequence token is needed to identify natural stopping."

print("Fresh student:", student_model_id)
print("Model revision:", resolved_student_revision)
print("Decoding: greedy; maximum new response tokens:", max_new_tokens)
print("No model weights have been updated in this notebook.")


---

<a id="baseline-generation"></a>

## 4. Generate and score responses

### 4.1 Keep the prompt and response separate

For each question, the tokenizer returns `input_ids` with shape
`[1, prompt_length]`. The model returns `[1, prompt_length + response_length]`.
As in notebook 01, only tokens after `prompt_length` belong to the response.

We check that the generated sequence begins with the original prompt, then decode
only the response. The answer checker must never inspect prompt text as if the
student generated it. We save both sets of token IDs for later inspection.

A **token-limit stop** means the response used all 512 available tokens without
an end token. It may be unfinished. That is a useful diagnosis, so we report it
alongside accuracy rather than treating it as an automatic excuse or success.


In [ ]:
def evaluate_one_example(example):
    """Generate one answer and check it without exposing the reference to the model."""
    prompt_text = build_evaluation_prompt(example["question"])
    assert prompt_text == example["prompt"]
    prompt_inputs = baseline_tokenizer(prompt_text, return_tensors="pt").to(baseline_device)
    prompt_token_ids = prompt_inputs["input_ids"]  # [1, prompt_length]
    prompt_length = prompt_token_ids.shape[1]
    assert prompt_length > 0 and prompt_token_ids.shape[0] == 1

    with torch.inference_mode():
        complete_sequence_ids = baseline_student.generate(
            **prompt_inputs, generation_config=baseline_generation_config,
        )
    # generate() returns the original question followed by newly generated tokens.
    response_token_ids = complete_sequence_ids[:, prompt_length:]
    response_length = response_token_ids.shape[1]
    assert torch.equal(complete_sequence_ids[:, :prompt_length], prompt_token_ids)
    assert 0 < response_length <= max_new_tokens
    assert response_length == complete_sequence_ids.shape[1] - prompt_length

    response_ids_list = response_token_ids[0].cpu().tolist()
    response_text = baseline_tokenizer.decode(response_ids_list, skip_special_tokens=True)
    # Only now do we consult the reference answer, outside the model input.
    predicted_answer = extract_final_answer(response_text)
    expected_answer = Decimal(example["expected_answer"])
    ended_with_eos = response_ids_list[-1] in eos_token_ids
    return {
        **example,
        "response_text": response_text,
        "predicted_answer": str(predicted_answer) if predicted_answer is not None else None,
        "answer_parsed": predicted_answer is not None,
        "is_correct": predicted_answer is not None and predicted_answer == expected_answer,
        "prompt_token_ids": prompt_token_ids[0].cpu().tolist(),
        "response_token_ids": response_ids_list,
        "prompt_length": prompt_length,
        "response_length": response_length,
        "ended_with_eos": ended_with_eos,
        "hit_token_limit": response_length == max_new_tokens and not ended_with_eos,
    }


### 4.2 Inspect the first response

We start with one question so that a tokenization, generation, or formatting
problem is easy to see. A wrong answer is a valid baseline observation; an
alignment error should stop the notebook. This result is reused in the full run,
so each selected question is generated only once.


In [ ]:
# This is the unchanged student answering its first held-out question.
evaluation_started_at = datetime.now(timezone.utc).isoformat()
evaluation_started_clock = time.perf_counter()
first_result = evaluate_one_example(evaluation_examples[0])
print("Question:", first_result["question"])
print("\nStudent response:\n" + first_result["response_text"])
print("\nParsed answer:", first_result["predicted_answer"])
print("Reference answer:", first_result["expected_answer"])
print("Correct:", first_result["is_correct"])
print("Response tokens:", first_result["response_length"])
print("Hit token limit:", first_result["hit_token_limit"])


### 4.3 Evaluate the remaining questions

We evaluate one prompt at a time. This keeps tensor shapes easy to follow and
avoids padding complications in this first baseline. Every selected question
stays in the results, including wrong and unparseable answers.


In [ ]:
# Keep the first result and generate each remaining response exactly once.
baseline_result_rows = [first_result]
for example_number, example in enumerate(evaluation_examples[1:], start=2):
    baseline_result_rows.append(evaluate_one_example(example))
    print(f"Evaluated {example_number}/{number_of_examples} questions")

evaluation_elapsed_seconds = time.perf_counter() - evaluation_started_clock
assert len(baseline_result_rows) == number_of_examples
assert [row["example_id"] for row in baseline_result_rows] == [
    example["example_id"] for example in evaluation_examples
]
for row in baseline_result_rows:
    # Recorded lengths and scores must remain consistent with the actual response.
    assert row["response_length"] == len(row["response_token_ids"])
    assert row["prompt_length"] == len(row["prompt_token_ids"])
    parsed_again = extract_final_answer(row["response_text"])
    assert row["answer_parsed"] == (parsed_again is not None)
    assert row["is_correct"] == (
        parsed_again is not None and parsed_again == Decimal(row["expected_answer"])
    )
assert not baseline_student.training
assert all(parameter.grad is None for parameter in baseline_student.parameters())
print("All baseline generation and scoring checks passed.")


---

<a id="baseline-metrics"></a>

## 5. Read the results

| Measurement | What it tells us |
| :--- | :--- |
| Final-answer accuracy | Fraction of **all** questions with a parsed, numerically correct answer |
| Answer-format success | Fraction that produced one readable marked final answer |
| Token-limit stops | How many responses may have run out of room before finishing |
| Mean response length | Average number of generated tokens, including an end token when present |

The main score never excludes formatting failures. The reference checks only the
final number, not the correctness of every reasoning step. With 32 questions,
one answer changes accuracy by about 3.1 percentage points, so small differences
are weak evidence. A zero score or a low format-success rate is possible for an
untuned base model and should be reported honestly.

We inspect a few failures to understand what happened. Do not tune the model or
the protocol repeatedly against these held-out examples; use separate validation
data for that. This set is meant for the later before/after comparison.


In [ ]:
# Accuracy and format success have the same denominator: every selected question.
baseline_summary = summarize_baseline_results(baseline_result_rows)
assert baseline_summary["number_of_examples"] == number_of_examples
assert 0 <= baseline_summary["accuracy"] <= baseline_summary["answer_parse_rate"] <= 1
baseline_results_table = pd.DataFrame(baseline_result_rows)
display(pd.DataFrame([baseline_summary]))
display(baseline_results_table[[
    "example_id", "expected_answer", "predicted_answer", "is_correct",
    "answer_parsed", "response_length", "hit_token_limit",
]].head(10))
print(f"Correct: {baseline_summary['number_correct']}/{number_of_examples}")
print(f"Accuracy: {baseline_summary['accuracy']:.1%}")
print(f"Answer-format success: {baseline_summary['answer_parse_rate']:.1%}")

# Read complete responses for a couple of failures; raw text is also saved below.
failed_results = [row for row in baseline_result_rows if not row["is_correct"]]
for failed_result in failed_results[:2]:
    print("\nExample:", failed_result["example_id"])
    print("Question:", failed_result["question"])
    print("Student response:\n" + failed_result["response_text"])
    print("Reference:", failed_result["expected_answer"])
    print("Parsed:", failed_result["predicted_answer"])


---

<a id="baseline-artifacts"></a>

## 6. Save and download the experiment

A score is much more useful when we know how it was produced. We save the model
and dataset revisions, selected question IDs, exact prompt template, generation
settings, answer-checking rule, precision, and package versions. A protocol hash
is a fingerprint of these evaluation rules and selected examples. The environment
is stored separately because GPU and package differences can affect results too.

| File | Contents |
| :--- | :--- |
| `manifest.json` | Evaluation protocol, exact selected questions, environment, and run information |
| `responses.jsonl` | One complete record per question, including text, token IDs, reference answer, and score |
| `results.csv` | A compact table for browsing or later analysis |
| `summary.json` | Accuracy, format success, and length diagnostics |

Each run gets a new folder under `results/`. In Colab this is local runtime
storage, not Google Drive. The ZIP contains all four files. Keep it outside Colab
so that a later model can be evaluated against the same protocol.

When repeating the original run, set `student_revision` and `dataset_revision`
to the exact commits from its manifest. For a later trained model, the model
identity will change; the question set, prompt, decoding policy, and scoring rule
should stay fixed. Compare those fields, rather than expecting the entire
model-inclusive protocol hash to remain identical.


In [ ]:
# File export is kept separate from GPU work so we can test it with small,
# made-up records before trusting it with real evaluation results.
def save_baseline_artifacts(results_root, run_id, manifest, result_rows, summary):
    """Save one complete run and return its folder and checked ZIP archive."""
    import csv

    assert len(result_rows) == manifest["protocol"]["number_of_examples"]
    assert summarize_baseline_results(result_rows) == summary
    run_directory = Path(results_root) / run_id
    run_directory.mkdir(parents=True, exist_ok=False)
    (run_directory / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8",
    )
    (run_directory / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    with (run_directory / "responses.jsonl").open("w", encoding="utf-8") as response_file:
        for row in result_rows:
            response_file.write(json.dumps(row, ensure_ascii=False) + "\n")

    # Keep large token-ID lists in JSONL; the CSV is a compact, readable table.
    csv_fields = [key for key in result_rows[0] if key not in {"prompt_token_ids", "response_token_ids"}]
    with (run_directory / "results.csv").open("w", encoding="utf-8", newline="") as csv_file:
        csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields, extrasaction="ignore")
        csv_writer.writeheader()
        csv_writer.writerows(result_rows)

    # Reading records back catches serialization mistakes or missing examples.
    saved_rows = [json.loads(line) for line in
                  (run_directory / "responses.jsonl").read_text(encoding="utf-8").splitlines()]
    assert saved_rows == result_rows
    assert summarize_baseline_results(saved_rows) == summary
    saved_manifest = json.loads((run_directory / "manifest.json").read_text(encoding="utf-8"))
    saved_protocol_json = json.dumps(saved_manifest["protocol"], sort_keys=True, ensure_ascii=False)
    assert hashlib.sha256(saved_protocol_json.encode("utf-8")).hexdigest() == manifest["protocol_sha256"]
    archive_path = Path(shutil.make_archive(str(run_directory), "zip", root_dir=run_directory))
    with zipfile.ZipFile(archive_path) as results_archive:
        assert results_archive.testzip() is None
        assert set(results_archive.namelist()) == {"manifest.json", "summary.json", "responses.jsonl", "results.csv"}
    return run_directory, archive_path


In [ ]:
# Record immutable source revisions and the exact prompts and references selected.
# Reference answers are saved for auditing; they were never sent to generate().
evaluation_protocol = {
    "schema_version": 1,
    "student_model_id": student_model_id,
    "student_revision": resolved_student_revision,
    "dataset_id": dataset_id,
    "dataset_config": dataset_config,
    "dataset_split": dataset_split,
    "dataset_revision": resolved_dataset_revision,
    "dataset_fingerprint": held_out_dataset._fingerprint,
    "selection_seed": evaluation_seed,
    "selected_indices": evaluation_indices,
    "number_of_examples": number_of_examples,
    "prompt_template": evaluation_prompt_template,
    "answer_checker": "single_marked_final_decimal_v1",
    "generation_config": baseline_generation_config.to_dict(),
    "examples": evaluation_examples,
}
protocol_json = json.dumps(evaluation_protocol, sort_keys=True, ensure_ascii=False)
protocol_sha256 = hashlib.sha256(protocol_json.encode("utf-8")).hexdigest()
run_id = datetime.now(timezone.utc).strftime("baseline_%Y%m%dT%H%M%S%fZ")
results_root = Path.cwd().parent / "results" if Path.cwd().name == "notebooks" else Path.cwd() / "results"

# Environment metadata explains differences that a random seed cannot prevent.
manifest = {
    "run_id": run_id,
    "status": "completed",
    "evaluation_started_at_utc": evaluation_started_at,
    "evaluation_elapsed_seconds": evaluation_elapsed_seconds,
    "protocol_sha256": protocol_sha256,
    "protocol": evaluation_protocol,
    "environment": {
        "python": platform.python_version(),
        "packages": {name: importlib.metadata.version(name) for name in [
            "torch", "transformers", "accelerate", "datasets", "pandas", "huggingface_hub",
        ]},
        "gpu": gpu_properties.name,
        "gpu_memory_bytes": gpu_properties.total_memory,
        "cuda_version": torch.version.cuda,
        "model_dtype": str(baseline_dtype),
        "seed": evaluation_seed,
        "attention_implementation": baseline_student.config._attn_implementation,
    },
}
# Write, read back, and package this complete run without changing any scores.
run_directory, archive_path = save_baseline_artifacts(
    results_root, run_id, manifest, baseline_result_rows, baseline_summary,
)
print("Protocol SHA256:", protocol_sha256)
print("Results folder:", run_directory)
print("Download archive:", archive_path)
print("All baseline artifact checks passed.")


### Download before disconnecting

With `download_results = True`, this cell offers the ZIP download in Colab. On a
local Jupyter installation it prints the file path instead. Also save or download
the executed `.ipynb` to preserve its visible output tables.


In [ ]:
# Colab's download helper transfers the finished archive to your computer.
# The file remains available in the Files panel if the browser blocks the download.
if download_results:
    try:
        from google.colab import files
    except ImportError:
        print("Outside Colab. Open the results archive at:", archive_path)
    else:
        files.download(str(archive_path))
else:
    print("Automatic download disabled. Save this archive before disconnecting:", archive_path)


---

<a id="baseline-recap"></a>

## What we achieved

**If the checks above pass, we have a recorded starting score for the original student.**

1. We loaded a fresh copy of the small model, without the update from notebook 01.
2. We asked the same fixed set of 32 questions using one consistent prompt and decoding rule.
3. We checked the final numeric answers. Wrong answers and unreadable answers both stayed in the score.
4. We recorded format failures and responses that ran out of tokens, so a low score is easier to understand.
5. We saved the questions, exact settings, raw responses, scores, and environment in a downloadable archive.

The teacher has not been used, and no model weights were updated in this notebook.
This evaluates the student's final answers. Notebook 01's reverse KL measures a
different thing: agreement with the teacher's next-token probabilities. Lower KL
and better task accuracy are not automatically the same outcome.

### Next checkpoint

Run this notebook in Colab, download the archive, and inspect the first response
and any failures. Then we can decide how to build the supervised cold start in
[`03_sft_cold_start.ipynb`](03_sft_cold_start.ipynb), using training data that does
not contain these held-out examples. Later, notebook 04 will use fresh student
responses for repeated distillation and compare the result against this baseline.

**In everyday terms: we gave the student a small test before teaching it, and
kept both the test and its answers so we can make a fair comparison later.**
